OUR DATA:

In [4]:
import pandas as pd
df = pd.read_csv("data.csv", sep = ";")
df

,loop_iteration,cycle_iteration,money,bet,bets in cycle,all-in,all-in-money,lost_cycle
0,1,1,10000,1,2,False,0,False
1,1,2,10001,1,5,False,0,False
2,1,3,10002,1,1,False,0,False
3,1,4,10003,1,1,False,0,False
4,1,5,10004,1,1,False,0,False
...,...,...,...,...,...,...,...,...
6275117,100,5,13996,999,2,False,0,False
6275118,100,6,14995,999,1,False,0,False
6275119,100,7,15994,999,5,True,1009,False
6275120,100,8,2018,999,1,False,0,False


lets try simplify this dataframe

In [2]:
data = []

for i in range(1,1000):
    for j in range(1,100):
        info = {
        "bet":i,
        "loop_iteration":j,
        "won":True
        }
        if True in df[(df["bet"]==i)&(df["loop_iteration"]==j)]["lost_cycle"].values:
            info["won"] = False
        data.append(info)
df2 = pd.DataFrame(data=data)
df2

,bet,loop_iteration,won
0,1,1,True
1,1,2,True
2,1,3,True
3,1,4,True
4,1,5,True
...,...,...,...
98896,999,95,False
98897,999,96,True
98898,999,97,True
98899,999,98,True


We cant do it like that becuase it took me around 28 minutes which is very suboptimal

In [11]:
df3 = df.groupby(by=["bet","loop_iteration"]).agg({
    "money": 'last',
    "all-in": 'sum',
    "cycle_iteration":'max',
    "lost_cycle":'sum',
    "bets in cycle": "mean"

}
)
df3.rename(columns={"lost_cycle":"lost_game", "bets in cycle":"avg_bets_in_cycle"},inplace=True)
df3['lost_game'].astype(bool)
df3

money  all-in  cycle_iteration  lost_game  \
bet loop_iteration                                              
1   1               19999       0            10000          0   
    2               12438       1             2439          1   
    3                6916       3             2880          1   
    4               10937       1              938          1   
    5               19999       0            10000          0   
...                   ...     ...              ...        ...   
999 96              19001       1               15          0   
    97              19990       0               11          0   
    98              19990       0               11          0   
    99              19001       1               13          0   
    100              3017       2                9          1   

                    avg_bets_in_cycle  
bet loop_iteration                     
1   1                        2.006700  
    2                        1.978270  
    3                        2.011458  
    4                        2.000000  
    5                        2.000500  
...                               ...  
999 96                       2.133333  
    97                       1.545455  
    98                       1.818182  
    99                       1.769231  
    100                      2.333333  

[99900 rows x 5 columns]

great, we got the same or even better(becuase we have more room to add unique columns with ease) results.

Let's check winratio for each bet.


In [30]:
#winratio overall

df_winratio = df3.groupby(by="bet").agg({
    "lost_game": "mean"
})
df_winratio.rename(columns={"lost_game":"win_ratio"},inplace=True)

df_winratio

,win_ratio
bet,
1,0.50
2,0.41
3,0.54
4,0.58
5,0.59
...,...
995,0.47
996,0.54
997,0.50


Ok, now let's try something harder.
How my implemented all-in strategy influence the expected value of money.

To calculate that we need information how many games we manage to rescue, and how much money we could save if we wouldn't use all-in strategy.


In [23]:

all_in_save = df[(df["lost_cycle"]==False) & (df["all-in"]==True) ][['bet', 'loop_iteration']]
all_in_save

,bet,loop_iteration
14090,1,3
14131,1,3
28991,1,6
34361,1,7
48109,1,8
...,...,...
6275001,999,90
6275032,999,93
6275063,999,96
6275102,999,99


Dataframe with bet and loop iteration of saved cycles by all in. Now we need to figure out how to check if the game was saved

In [27]:
df3[(df3["lost_game"]==True)]

money  all-in  cycle_iteration  lost_game  \
bet loop_iteration                                              
1   2               12438       1             2439          1   
    3                6916       3             2880          1   
    4               10937       1              938          1   
    6               12143       2             5793          1   
    8               11544       2             5941          1   
...                   ...     ...              ...        ...   
999 79              10999       1                2          1   
    85              12997       1                4          1   
    88              19990       1               11          1   
    94              13996       1                5          1   
    100              3017       2                9          1   

                    avg_bets_in_cycle  
bet loop_iteration                     
1   2                        1.978270  
    3                        2.011458  
    4                        2.000000  
    6                        2.004833  
    8                        1.976772  
...                               ...  
999 79                       2.500000  
    85                       2.250000  
    88                       2.090909  
    94                       1.800000  
    100                      2.333333  

[50096 rows x 5 columns]